In [21]:
import random
import time
from collections import deque
import ipywidgets as widgets
from IPython.display import display

class BuscaminasEnv:
    def __init__(self, filas=10, cols=10, minas=10, semilla=42):
        self.filas = filas
        self.cols = cols
        self.num_minas = minas
        self.semilla = semilla
        self.rng = random.Random(semilla)

        self.tablero_real = [[0 for _ in range(cols)] for _ in range(filas)]
        self.visible = [['?' for _ in range(cols)] for _ in range(filas)]
        self.primer_movimiento = True
        self.terminado = False
        self.victoria = False
        self.recompensa_total = 0

    def _generar_tablero(self, fila_segura, col_segura):
        # Excluye la primera casilla y sus 8 vecinas para asegurar una apertura limpia
        posiciones = [
            (r, c) for r in range(self.filas) for c in range(self.cols)
            if abs(r - fila_segura) > 1 or abs(c - col_segura) > 1
        ]
        if len(posiciones) < self.num_minas:
            posiciones = [
                (r, c) for r in range(self.filas) for c in range(self.cols)
                if (r, c) != (fila_segura, col_segura)
            ]

        minas_pos = self.rng.sample(posiciones, self.num_minas)
        for r, c in minas_pos:
            self.tablero_real[r][c] = -1

        for r in range(self.filas):
            for c in range(self.cols):
                if self.tablero_real[r][c] == -1:
                    continue
                self.tablero_real[r][c] = sum(
                    1 for nr, nc in self.obtener_vecinos(r, c)
                    if self.tablero_real[nr][nc] == -1
                )

    def obtener_vecinos(self, r, c):
        vecinos = []
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.filas and 0 <= nc < self.cols:
                    vecinos.append((nr, nc))
        return vecinos

    def revelar(self, r, c):
        if self.terminado or self.visible[r][c] != '?':
            return

        if self.primer_movimiento:
            self._generar_tablero(r, c)
            self.primer_movimiento = False

        if self.tablero_real[r][c] == -1:
            self.visible[r][c] = 'X'
            self.terminado = True
            self.victoria = False
            self.recompensa_total -= 100
            return

        cola = deque([(r, c)])
        while cola:
            curr_r, curr_c = cola.popleft()
            if self.visible[curr_r][curr_c] != '?':
                continue
            val = self.tablero_real[curr_r][curr_c]
            self.visible[curr_r][curr_c] = str(val)
            self.recompensa_total += 1

            if val == 0:
                for nr, nc in self.obtener_vecinos(curr_r, curr_c):
                    if self.visible[nr][nc] == '?':
                        cola.append((nr, nc))

        self._verificar_victoria()

    def alternar_bandera(self, r, c):
        if self.terminado:
            return
        if self.visible[r][c] == '?':
            self.visible[r][c] = 'F'
        elif self.visible[r][c] == 'F':
            self.visible[r][c] = '?'

    def marcar_mina(self, r, c):
        if not self.terminado and self.visible[r][c] == '?':
            self.visible[r][c] = 'F'

    def _verificar_victoria(self):
        ocultas_o_banderas = sum(
            1 for r in range(self.filas) for c in range(self.cols)
            if self.visible[r][c] in ['?', 'F']
        )
        if ocultas_o_banderas == self.num_minas:
            self.terminado = True
            self.victoria = True
            self.recompensa_total += 100

    def obtener_ocultas(self):
        return [(r, c) for r in range(self.filas) for c in range(self.cols) if self.visible[r][c] == '?']

    def obtener_numeros_frontera(self):
        frontera = []
        for r in range(self.filas):
            for c in range(self.cols):
                if self.visible[r][c].isdigit() and int(self.visible[r][c]) > 0:
                    vecinos = self.obtener_vecinos(r, c)
                    if any(self.visible[nr][nc] == '?' for nr, nc in vecinos):
                        frontera.append((r, c))
        return frontera

class AgenteBase:
    def __init__(self, semilla=42):
        self.rng = random.Random(semilla)
        self.metricas = {"reglas_evaluadas": 0, "reglas_disparadas": 0, "hechos_inferidos": 0, "jugadas_azar": 0}
        self.ultimo_paso = {"seguras": [], "minas": [], "azar": [], "regla": "Esperando inicio..."}

In [22]:
class AgenteForwardChaining(AgenteBase):
    def paso(self, env: BuscaminasEnv):
        if env.primer_movimiento:
            r, c = self.rng.choice(env.obtener_ocultas())
            self.metricas["jugadas_azar"] += 1
            self.ultimo_paso = {"seguras": [(r, c)], "minas": [], "azar": [], "regla": f"Primera jugada segura en ({r}, {c})"}
            env.revelar(r, c)
            return

        seguras_inf = set()
        minas_inf = set()
        reglas_texto = []

        hubo_cambios = True
        while hubo_cambios:
            hubo_cambios = False
            for r, c in env.obtener_numeros_frontera():
                self.metricas["reglas_evaluadas"] += 2
                num = int(env.visible[r][c])
                vecinos = env.obtener_vecinos(r, c)
                ocultas = [v for v in vecinos if env.visible[v[0]][v[1]] == '?' and v not in minas_inf and v not in seguras_inf]
                banderas = [v for v in vecinos if env.visible[v[0]][v[1]] == 'F' or v in minas_inf]
                minas_rest = num - len(banderas)

                if not ocultas:
                    continue

                # Regla 1: Minas restantes == Casillas ocultas -> Todas son MINAS
                if minas_rest == len(ocultas):
                    self.metricas["reglas_disparadas"] += 1
                    for o in ocultas:
                        minas_inf.add(o)
                        self.metricas["hechos_inferidos"] += 1
                    reglas_texto.append(f"R1 en ({r},{c}) [num={num}]: {len(ocultas)} ocultas = MINAS")
                    hubo_cambios = True

                # Regla 2: Minas restantes == 0 -> Todas son SEGURAS
                elif minas_rest == 0:
                    self.metricas["reglas_disparadas"] += 1
                    for o in ocultas:
                        seguras_inf.add(o)
                        self.metricas["hechos_inferidos"] += 1
                    reglas_texto.append(f"R2 en ({r},{c}) [num={num}]: {len(ocultas)} ocultas = SEGURAS")
                    hubo_cambios = True

        for mr, mc in minas_inf:
            env.marcar_mina(mr, mc)

        if seguras_inf or minas_inf:
            for sr, sc in seguras_inf:
                env.revelar(sr, sc)
            self.ultimo_paso = {
                "seguras": list(seguras_inf),
                "minas": list(minas_inf),
                "azar": [],
                "regla": " | ".join(reglas_texto[:2]) + ("..." if len(reglas_texto) > 2 else "")
            }
            return

        # Decisión estrictamente aleatoria si se atasca
        ocultas = env.obtener_ocultas()
        if ocultas:
            r, c = self.rng.choice(ocultas)
            self.metricas["jugadas_azar"] += 1
            self.ultimo_paso = {"seguras": [], "minas": [], "azar": [(r, c)], "regla": f"Sin deducciones -> Azar puro en ({r}, {c})"}
            env.revelar(r, c)

In [23]:
class AgenteBackwardChaining(AgenteBase):
    def demostrar_estado(self, env: BuscaminasEnv, objetivo, estado_hipotetico, visitados=None):
        if visitados is None:
            visitados = set()
        if (objetivo, estado_hipotetico) in visitados:
            return False
        visitados.add((objetivo, estado_hipotetico))

        r, c = objetivo
        if env.visible[r][c] == 'F':
            return estado_hipotetico == 'MINA'
        if env.visible[r][c].isdigit():
            return estado_hipotetico == 'SEGURA'

        vecinos_num = [v for v in env.obtener_vecinos(r, c) if env.visible[v[0]][v[1]].isdigit() and int(env.visible[v[0]][v[1]]) > 0]
        for nr, nc in vecinos_num:
            self.metricas["reglas_evaluadas"] += 1
            num = int(env.visible[nr][nc])
            otros = [v for v in env.obtener_vecinos(nr, nc) if v != objetivo]
            banderas = [v for v in otros if env.visible[v[0]][v[1]] == 'F']
            ocultas = [v for v in otros if env.visible[v[0]][v[1]] == '?']

            if estado_hipotetico == 'SEGURA':
                if len(banderas) == num:
                    self.metricas["reglas_disparadas"] += 1
                    return True
                if len(banderas) + len(ocultas) == num:
                    if all(self.demostrar_estado(env, ov, 'MINA', visitados.copy()) for ov in ocultas):
                        self.metricas["reglas_disparadas"] += 1
                        return True
            elif estado_hipotetico == 'MINA':
                faltantes = num - len(banderas)
                if faltantes == len(ocultas) + 1:
                    self.metricas["reglas_disparadas"] += 1
                    return True
                no_seguras = [ov for ov in ocultas if not self.demostrar_estado(env, ov, 'SEGURA', visitados.copy())]
                if faltantes == len(no_seguras) + 1:
                    self.metricas["reglas_disparadas"] += 1
                    return True
        return False

    def paso(self, env: BuscaminasEnv):
        if env.primer_movimiento:
            r, c = self.rng.choice(env.obtener_ocultas())
            self.metricas["jugadas_azar"] += 1
            self.ultimo_paso = {"seguras": [(r, c)], "minas": [], "azar": [], "regla": f"Primera jugada segura en ({r}, {c})"}
            env.revelar(r, c)
            return

        candidatas = list({v for r, c in env.obtener_numeros_frontera() for v in env.obtener_vecinos(r, c) if env.visible[v[0]][v[1]] == '?'})
        self.rng.shuffle(candidatas)

        for cand in candidatas:
            if self.demostrar_estado(env, cand, 'MINA'):
                self.metricas["hechos_inferidos"] += 1
                env.marcar_mina(cand[0], cand[1])
                self.ultimo_paso = {"seguras": [], "minas": [cand], "azar": [], "regla": f"Hipótesis {cand} == MINA demostrada hacia atrás"}
                return
            if self.demostrar_estado(env, cand, 'SEGURA'):
                self.metricas["hechos_inferidos"] += 1
                env.revelar(cand[0], cand[1])
                self.ultimo_paso = {"seguras": [cand], "minas": [], "azar": [], "regla": f"Hipótesis {cand} == SEGURA demostrada hacia atrás"}
                return

        # Decisión estrictamente aleatoria si ninguna hipótesis se demuestra
        ocultas = env.obtener_ocultas()
        if ocultas:
            r, c = self.rng.choice(ocultas)
            self.metricas["jugadas_azar"] += 1
            self.ultimo_paso = {"seguras": [], "minas": [], "azar": [(r, c)], "regla": f"Ninguna hipótesis demostrada -> Azar en ({r}, {c})"}
            env.revelar(r, c)

In [24]:
class AgenteMonteCarlo(AgenteBase):
    def __init__(self, simulaciones=300, semilla=42):
        super().__init__(semilla)
        self.simulaciones = simulaciones

    def _generar_asignacion_valida(self, frontera, restricciones, minas_restantes):
        asignacion = {}
        orden = list(frontera)
        self.rng.shuffle(orden)

        def backtrack(idx, usadas):
            if usadas > minas_restantes:
                return False
            if idx == len(orden):
                return all(sum(asignacion[c] for c in celdas) == req for celdas, req in restricciones)
            celda = orden[idx]
            opciones = [0, 1]
            self.rng.shuffle(opciones)
            for val in opciones:
                asignacion[celda] = val
                valido = True
                for celdas, req in restricciones:
                    if celda in celdas:
                        asig = [asignacion[c] for c in celdas if c in asignacion]
                        if sum(asig) > req or sum(asig) + (len(celdas) - len(asig)) < req:
                            valido = False
                            break
                if valido and backtrack(idx + 1, usadas + val):
                    return True
                del asignacion[celda]
            return False

        return asignacion if backtrack(0, 0) else None

    def paso(self, env: BuscaminasEnv):
        if env.primer_movimiento:
            r, c = self.rng.choice(env.obtener_ocultas())
            self.metricas["jugadas_azar"] += 1
            self.ultimo_paso = {"seguras": [(r, c)], "minas": [], "azar": [], "regla": f"Primera jugada segura en ({r}, {c})"}
            env.revelar(r, c)
            return

        frontera_set = set()
        restricciones = []
        banderas_tot = sum(1 for r in range(env.filas) for c in range(env.cols) if env.visible[r][c] == 'F')
        minas_rest = env.num_minas - banderas_tot

        for r, c in env.obtener_numeros_frontera():
            num = int(env.visible[r][c])
            vecinos = env.obtener_vecinos(r, c)
            ocultas = [v for v in vecinos if env.visible[v[0]][v[1]] == '?']
            banderas = sum(1 for v in vecinos if env.visible[v[0]][v[1]] == 'F')
            if ocultas:
                frontera_set.update(ocultas)
                restricciones.append((ocultas, num - banderas))

        frontera = list(frontera_set)
        if not frontera:
            ocultas = env.obtener_ocultas()
            if ocultas:
                r, c = self.rng.choice(ocultas)
                self.metricas["jugadas_azar"] += 1
                self.ultimo_paso = {"seguras": [], "minas": [], "azar": [(r, c)], "regla": f"Sin frontera -> Azar puro en ({r}, {c})"}
                env.revelar(r, c)
            return

        conteo = {c: 0 for c in frontera}
        validas = 0
        for _ in range(self.simulaciones):
            self.metricas["reglas_evaluadas"] += 1
            m = self._generar_asignacion_valida(frontera, restricciones, minas_rest)
            if m is not None:
                validas += 1
                for c, es_m in m.items():
                    conteo[c] += es_m

        if validas > 0:
            self.metricas["reglas_disparadas"] += 1
            minas_mc = [c for c, cnt in conteo.items() if cnt == validas]
            seguras_mc = [c for c, cnt in conteo.items() if cnt == 0]

            for mr, mc in minas_mc:
                env.marcar_mina(mr, mc)
                self.metricas["hechos_inferidos"] += 1

            if seguras_mc or minas_mc:
                for sr, sc in seguras_mc:
                    self.metricas["hechos_inferidos"] += 1
                    env.revelar(sr, sc)
                self.ultimo_paso = {
                    "seguras": seguras_mc,
                    "minas": minas_mc,
                    "azar": [],
                    "regla": f"Monte Carlo ({validas} muestras válidas): {len(seguras_mc)} seguras (0%), {len(minas_mc)} minas (100%)"
                }
                return

            mejor = min(conteo.keys(), key=lambda c: conteo[c])
            prob = (conteo[mejor] / validas) * 100
            self.metricas["jugadas_azar"] += 1
            self.ultimo_paso = {
                "seguras": [],
                "minas": [],
                "azar": [mejor],
                "regla": f"Monte Carlo ({validas} muestras): menor riesgo en {mejor} ({prob:.1f}% mina)"
            }
            env.revelar(mejor[0], mejor[1])
        else:
            r, c = self.rng.choice(env.obtener_ocultas())
            self.metricas["jugadas_azar"] += 1
            self.ultimo_paso = {"seguras": [], "minas": [], "azar": [(r, c)], "regla": f"Sin muestras válidas -> Azar en ({r}, {c})"}
            env.revelar(r, c)

In [ ]:
DIFICULTADES_POR_DEFECTO = [
    ("Principiante", 10, 10, 10),
    ("Intermedio", 16, 16, 40),
    ("Experto", 16, 30, 99),
]

def evaluar_agentes(n_partidas=50, dificultades=None):
    if dificultades is None:
        dificultades = DIFICULTADES_POR_DEFECTO

    agentes_cls = {
        "1. Forward Chaining (+ Azar)": lambda s: AgenteForwardChaining(semilla=s),
        "2. Backward Chaining (+ Azar)": lambda s: AgenteBackwardChaining(semilla=s),
        "3. Monte Carlo (N=300)": lambda s: AgenteMonteCarlo(simulaciones=300, semilla=s),
    }

    for etiqueta, filas, cols, minas in dificultades:
        resultados = {
            nombre: {"victorias": 0, "tiempo": 0.0, "hechos": 0, "azar": 0}
            for nombre in agentes_cls
        }

        for i in range(n_partidas):
            semilla_tablero = 1000 + i
            for nombre, constructor in agentes_cls.items():
                env = BuscaminasEnv(filas=filas, cols=cols, minas=minas, semilla=semilla_tablero)
                agente = constructor(semilla_tablero)

                t0 = time.perf_counter()
                while not env.terminado:
                    agente.paso(env)
                t1 = time.perf_counter()

                resultados[nombre]["victorias"] += int(env.victoria)
                resultados[nombre]["tiempo"] += (t1 - t0)
                resultados[nombre]["hechos"] += agente.metricas["hechos_inferidos"]
                resultados[nombre]["azar"] += agente.metricas["jugadas_azar"]

        print(f"\n=== BENCHMARK {etiqueta.upper()} SOBRE {n_partidas} TABLEROS ({filas}x{cols}, {minas} minas) ===")
        for nombre, res in resultados.items():
            tasa = (res["victorias"] / n_partidas) * 100
            t_prom = (res["tiempo"] / n_partidas) * 1000
            print(
                f"{nombre:30s} | Victorias: {tasa:5.1f}% | "
                f"Tiempo/partida: {t_prom:6.2f} ms | "
                f"Hechos inf.: {res['hechos']/n_partidas:5.1f} | "
                f"Jugadas azar/riesgo: {res['azar']/n_partidas:4.1f}"
            )

# Ejecutar prueba en todas las dificultades
evaluar_agentes(n_partidas=10)


=== BENCHMARK PRINCIPIANTE SOBRE 10 TABLEROS (10x10, 10 minas) ===
1. Forward Chaining (+ Azar)   | Victorias:  90.0% | Tiempo/partida:   2.16 ms | Hechos inf.:  25.6 | Jugadas azar/riesgo:  1.2
2. Backward Chaining (+ Azar)  | Victorias:  80.0% | Tiempo/partida:  23.72 ms | Hechos inf.:  21.6 | Jugadas azar/riesgo:  1.2
3. Monte Carlo (N=300)         | Victorias: 100.0% | Tiempo/partida: 615.62 ms | Hechos inf.:  29.5 | Jugadas azar/riesgo:  1.0


In [26]:
COLORES_NUMEROS = {
    '1': '#1976D2', '2': '#388E3C', '3': '#D32F2F', '4': '#7B1FA2',
    '5': '#FF8F00', '6': '#0097A7', '7': '#212121', '8': '#757575'
}

class InterfazBuscaminasColab:
    def __init__(self):
        self.dd_dificultad = widgets.Dropdown(
            options=[
                ('Principiante (10x10, 10 minas)', (10, 10, 10)),
                ('Intermedio (16x16, 40 minas)', (16, 16, 40)),
                ('Experto (16x30, 99 minas)', (16, 30, 99))
            ],
            value=(10, 10, 10),
            description='Tablero:'
        )
        self.dd_algoritmo = widgets.Dropdown(
            options=[
                ('1. Encadenamiento hacia Adelante', 'forward'),
                ('2. Encadenamiento hacia Atrás', 'backward'),
                ('3. Monte Carlo (N=300)', 'montecarlo'),
                ('Modo Manual (Clic en celdas)', 'manual')
            ],
            value='forward',
            description='Agente:'
        )
        self.txt_semilla = widgets.IntText(value=42, description='Semilla:', layout=widgets.Layout(width='150px'))
        self.sld_velocidad = widgets.FloatSlider(value=0.25, min=0.05, max=1.5, step=0.05, description='Pausa (s):')
        self.chk_bandera = widgets.Checkbox(value=False, description='Clic pone/quita Bandera (Modo Manual)')

        self.btn_reiniciar = widgets.Button(description='Nuevo / Reiniciar', button_style='info', icon='refresh')
        self.btn_paso = widgets.Button(description='Paso a paso', button_style='primary', icon='step-forward')
        self.btn_auto = widgets.Button(description='Ejecutar Auto', button_style='success', icon='play')
        self.btn_detener = widgets.Button(description='Detener', button_style='warning', icon='stop')

        self.lbl_estado = widgets.HTML()
        self.lbl_regla = widgets.HTML()
        self.lbl_metricas = widgets.HTML()
        self.leyenda_html = widgets.HTML(
            value="<div style='margin: 6px 0; font-size: 13px;'>"
                  "<span style='background:#A5D6A7; padding:2px 8px; border-radius:4px; margin-right:8px;'><b>Verde:</b> Deducida Segura</span>"
                  "<span style='background:#EF9A9A; padding:2px 8px; border-radius:4px; margin-right:8px;'><b>Rojo:</b> Deducida Mina</span>"
                  "<span style='background:#FFCC80; padding:2px 8px; border-radius:4px;'><b>Naranja:</b> Jugada Azar / Riesgo</span>"
                  "</div>"
        )
        self.contenedor_tablero = widgets.VBox()
        self.botones_celdas = []
        self.detener_solicitado = False

        self.btn_reiniciar.on_click(lambda _: self.reiniciar())
        self.btn_paso.on_click(lambda _: self.ejecutar_un_paso())
        self.btn_auto.on_click(lambda _: self.ejecutar_automatico())
        self.btn_detener.on_click(lambda _: self._solicitar_detener())

        self.reiniciar()

    def _solicitar_detener(self):
        self.detener_solicitado = True

    def reiniciar(self):
        self.detener_solicitado = True
        filas, cols, minas = self.dd_dificultad.value
        semilla = self.txt_semilla.value
        self.env = BuscaminasEnv(filas=filas, cols=cols, minas=minas, semilla=semilla)

        algo = self.dd_algoritmo.value
        if algo == 'forward':
            self.agente = AgenteForwardChaining(semilla=semilla)
        elif algo == 'backward':
            self.agente = AgenteBackwardChaining(semilla=semilla)
        elif algo == 'montecarlo':
            self.agente = AgenteMonteCarlo(simulaciones=300, semilla=semilla)
        else:
            self.agente = AgenteBase(semilla=semilla)

        self._construir_cuadricula(filas, cols)
        self.lbl_estado.value = f"<b>Estado:</b> En curso ({filas}x{cols}, {minas} minas) | Semilla: <code>{semilla}</code>"
        self.lbl_regla.value = "<b>Última acción:</b> Listo. La primera jugada siempre es segura."
        self.actualizar_metricas()
        self.actualizar_tablero()

    def _construir_cuadricula(self, filas, cols):
        self.botones_celdas = []
        filas_widgets = []
        tam_celda = '28px' if cols <= 16 else '24px'

        for r in range(filas):
            fila_btns = []
            for c in range(cols):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width=tam_celda, height=tam_celda, padding='0px', margin='1px')
                )
                btn.on_click(lambda _, row=r, col=c: self._clic_celda(row, col))
                fila_btns.append(btn)
            self.botones_celdas.append(fila_btns)
            filas_widgets.append(widgets.HBox(fila_btns, layout=widgets.Layout(margin='0px', padding='0px')))

        self.contenedor_tablero.children = tuple(filas_widgets)

    def _clic_celda(self, r, c):
        if self.dd_algoritmo.value != 'manual' or self.env.terminado:
            return
        if self.chk_bandera.value:
            self.env.alternar_bandera(r, c)
            self.agente.ultimo_paso = {"seguras": [], "minas": [(r, c)], "azar": [], "regla": f"Manual: Bandera alternada en ({r}, {c})"}
        else:
            es_primera = self.env.primer_movimiento
            self.env.revelar(r, c)
            msg = f"Manual: Primera jugada segura en ({r}, {c})" if es_primera else f"Manual: Revelada casilla ({r}, {c})"
            self.agente.ultimo_paso = {"seguras": [(r, c)], "minas": [], "azar": [], "regla": msg}

        self.lbl_regla.value = f"<b>Última acción:</b> {self.agente.ultimo_paso['regla']}"
        self.verificar_fin()
        self.actualizar_metricas()
        self.actualizar_tablero()

    def actualizar_metricas(self):
        m = self.agente.metricas
        banderas = sum(1 for r in range(self.env.filas) for c in range(self.env.cols) if self.env.visible[r][c] == 'F')
        self.lbl_metricas.value = (
            f"<b>Métricas:</b> Minas restantes aprox: <code>{self.env.num_minas - banderas}</code> | "
            f"Hechos inferidos: <code>{m['hechos_inferidos']}</code> | "
            f"Reglas disp.: <code>{m['reglas_disparadas']}</code> | "
            f"Jugadas azar/riesgo: <code>{m['jugadas_azar']}</code> | "
            f"Recompensa: <code>{self.env.recompensa_total}</code>"
        )

    def ejecutar_un_paso(self):
        if self.env.terminado or self.dd_algoritmo.value == 'manual':
            return
        self.agente.paso(self.env)
        self.lbl_regla.value = f"<b>Última acción:</b> {self.agente.ultimo_paso['regla']}"
        self.verificar_fin()
        self.actualizar_metricas()
        self.actualizar_tablero()

    def ejecutar_automatico(self):
        if self.env.terminado or self.dd_algoritmo.value == 'manual':
            return
        self.detener_solicitado = False
        while not self.env.terminado and not self.detener_solicitado:
            self.ejecutar_un_paso()
            time.sleep(self.sld_velocidad.value)

    def verificar_fin(self):
        if self.env.terminado:
            self.detener_solicitado = True
            if self.env.victoria:
                self.lbl_estado.value = "<b style='color:#2E7D32;'>Estado: ¡VICTORIA! Todas las casillas seguras reveladas.</b>"
            else:
                self.lbl_estado.value = "<b style='color:#C62828;'>Estado: DERROTA. Se detonó una mina.</b>"

    def actualizar_tablero(self):
        seguras = set(self.agente.ultimo_paso.get("seguras", []))
        minas = set(self.agente.ultimo_paso.get("minas", []))
        azar = set(self.agente.ultimo_paso.get("azar", []))

        for r in range(self.env.filas):
            for c in range(self.env.cols):
                btn = self.botones_celdas[r][c]
                val = self.env.visible[r][c]

                if self.env.terminado and not self.env.victoria and self.env.tablero_real[r][c] == -1 and val != 'X':
                    val = '*'

                if val == '?':
                    btn.description = ''
                elif val == 'F':
                    btn.description = 'F'
                elif val in ['X', '*']:
                    btn.description = 'M'
                elif val == '0':
                    btn.description = ' '
                else:
                    btn.description = val

                if (r, c) in seguras:
                    btn.style.button_color = '#A5D6A7'
                elif (r, c) in minas:
                    btn.style.button_color = '#EF9A9A'
                elif (r, c) in azar:
                    btn.style.button_color = '#FFCC80'
                elif val in ['?', 'F']:
                    btn.style.button_color = '#CFD8DC'
                elif val == 'X':
                    btn.style.button_color = '#E53935'
                elif val == '*':
                    btn.style.button_color = '#FFCDD2'
                else:
                    btn.style.button_color = '#F5F5F5'

                if val in COLORES_NUMEROS:
                    btn.style.text_color = COLORES_NUMEROS[val]
                elif val == 'F':
                    btn.style.text_color = '#B71C1C'
                elif val in ['X', '*']:
                    btn.style.text_color = '#000000'
                else:
                    btn.style.text_color = '#212121'

    def mostrar(self):
        fila_1 = widgets.HBox([self.dd_dificultad, self.dd_algoritmo, self.txt_semilla])
        fila_2 = widgets.HBox([self.btn_reiniciar, self.btn_paso, self.btn_auto, self.btn_detener, self.sld_velocidad])
        fila_3 = widgets.HBox([self.chk_bandera])
        panel = widgets.VBox([
            fila_1, fila_2, fila_3,
            self.lbl_estado, self.lbl_regla, self.lbl_metricas,
            self.leyenda_html, self.contenedor_tablero
        ])
        display(panel)

ui = InterfazBuscaminasColab()
ui.mostrar()